# Simulating a Hawkes process

The answers to Section 6 of the sample questions, `documentation/tex/questions`, run.
The questions concern Section 1.3.5 of the lecture notes: thinning, the update of the state,
and the exact scheme for the exponential kernel, with the simulators of `unito26.lob.hawkes`.
The two types are read as the market buys (type 0) and the market sells (type 1).

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
A Monte Carlo choice is decided against the standard error of the count, from
`HawkesParams.count_covariance_rate`: a rate within four standard errors of the stationary
rate is close to it, and one beyond ten is clearly away from it.

In [ ]:
import contextlib
import io
import timeit
from itertools import islice

import numpy as np
import pandas as pd
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")


def standard_error_of_rate(params, horizon):
    """The standard error of N(T)/T over a long horizon: sqrt(1' C 1 / T)."""
    return np.sqrt(params.count_covariance_rate().sum() / horizon)


class Uniforms:
    """Stands in for the generator: returns the given uniforms in order."""

    def __init__(self, values):
        self.values = list(values)

    def random(self, size=None):
        if size is None:
            return self.values.pop(0)
        return np.array([self.values.pop(0) for _ in range(size)])

## `question.thinningBoundAfterTheJump`: a bound never tightened, and a bound read before the jump

Alternative A keeps the post-jump bound until the next event; Alternative B tightens the bound
at every candidate, the accepted one included, before its jump.

In [ ]:
def thinning_rate(params, horizon, rng):
    mu_bar = params.baseline.sum()
    jumps = params.excitation.sum(axis=0)
    time, state, count = 0.0, np.zeros(2), 0
    bound = mu_bar
    while True:
        wait = rng.exponential(1 / bound)
        time += wait
        if time > horizon:
            return count / horizon
        state *= np.exp(-params.decay * wait)
        intensity = mu_bar + jumps @ state
        if rng.random() * bound <= intensity:
            lam = params.baseline + params.excitation @ state
            state[rng.choice(2, p=lam / lam.sum())] += 1.0
            count += 1
            bound = mu_bar + jumps @ state

thinning_rate_a = thinning_rate

In [ ]:
def thinning_rate(params, horizon, rng):
    mu_bar = params.baseline.sum()
    jumps = params.excitation.sum(axis=0)
    time, state, count = 0.0, np.zeros(2), 0
    bound = mu_bar
    while True:
        wait = rng.exponential(1 / bound)
        time += wait
        if time > horizon:
            return count / horizon
        state *= np.exp(-params.decay * wait)
        intensity = mu_bar + jumps @ state
        if rng.random() * bound <= intensity:
            lam = params.baseline + params.excitation @ state
            state[rng.choice(2, p=lam / lam.sum())] += 1.0
            count += 1
        bound = intensity

thinning_rate_b = thinning_rate

In [ ]:
def run(thinning_rate):
    params = config.self_exciting_pair_params()
    rng = np.random.default_rng(8)
    print(round(thinning_rate(params, 2e4, rng), 2))


params = config.self_exciting_pair_params()
total_rate = params.stationary_intensity().sum()
error = standard_error_of_rate(params, 2e4)
a, b = float(outcome(run, thinning_rate_a)), float(outcome(run, thinning_rate_b))
print(f"stationary rate {total_rate:.4f}, standard error of N/T {error:.4f}")
print(f"A: {a}, {(a - total_rate) / error:+.1f} SE;   B: {b}, {(b - total_rate) / error:+.1f} SE")
close = lambda rate: abs(rate - total_rate) < 4 * error
below = lambda rate: rate < total_rate - 10 * error
above = lambda rate: rate > total_rate + 10 * error
verify("question.thinningBoundAfterTheJump", "D", [
    close(a) and close(b),
    below(a) and close(b),
    close(a) and above(b),
    close(a) and below(b),
])

## `question.recordedStatesAliasTheJump`: a record that is the state itself

In [ ]:
def recorded_states(decay, times, types):
    state, previous, record = np.zeros(2), 0.0, []
    for now, event_type in zip(times, types):
        state = state * np.exp(-decay * (now - previous))
        record.append(state)
        state[event_type] += 1.0
        previous = now
    return record

recorded_states_a = recorded_states

In [ ]:
def recorded_states(decay, times, types):
    state, previous, record = np.zeros(2), 0.0, []
    for now, event_type in zip(times, types):
        state *= np.exp(-decay * (now - previous))
        record.append(state)
        state[event_type] += 1.0
        previous = now
    return record

recorded_states_b = recorded_states

In [ ]:
def run(recorded_states):
    times, types = [0.5, 1.0, 1.5], [0, 1, 0]
    record = recorded_states(2 * np.log(2), times, types)
    print([state.tolist() for state in record])


assert np.exp(-2 * np.log(2) * 0.5) == 0.5
a, b = outcome(run, recorded_states_a), outcome(run, recorded_states_b)
print("A:", a)
print("B:", b)
pre_jump = "[[0.0, 0.0], [0.5, 0.0], [0.25, 0.5]]"
post_jump = "[[1.0, 0.0], [0.5, 1.0], [1.25, 0.5]]"
last_thrice = "[[1.25, 0.5], [1.25, 0.5], [1.25, 0.5]]"
verify("question.recordedStatesAliasTheJump", "C", [
    a == pre_jump and b == last_thrice,
    a == pre_jump and b == pre_jump,
    a == post_jump and b == last_thrice,
    a == post_jump and b == post_jump,
])

## `question.intensityByRecursionAgainstSum`: the state carried against the history summed

Both functions are timed on the first $n$ events of one path of the asymmetric pair, for $n$
from 1000 to 16000, after a check that they agree.  The choices are read off the time per
event of each, from the bottom of the ladder to its top, and off the ratio of the two.

In [ ]:
def total_intensities_a(params, times, types):
    jumps = params.excitation.sum(axis=0)
    out = np.empty(times.size)
    for n, now in enumerate(times):
        weights = np.exp(-params.decay * (now - times[:n]))
        out[n] = params.baseline.sum() + np.sum(weights * jumps[types[:n]])
    return out

In [ ]:
def total_intensities_b(params, times, types):
    jumps = params.excitation.sum(axis=0)
    state, previous = np.zeros(2), 0.0
    out = np.empty(times.size)
    for n, (now, event_type) in enumerate(zip(times, types)):
        state *= np.exp(-params.decay * (now - previous))
        out[n] = params.baseline.sum() + jumps @ state
        state[event_type] += 1.0
        previous = now
    return out

In [ ]:
params = config.asymmetric_pair_params()
path = list(islice(ExponentialHawkes(params, rng=4).events(np.inf), 16_000))
path_times = np.array([time for time, _ in path])
path_types = np.array([event_type for _, event_type in path])
np.testing.assert_allclose(
    total_intensities_a(params, path_times[:4000], path_types[:4000]),
    total_intensities_b(params, path_times[:4000], path_types[:4000]),
    rtol=1e-9,
)


def microseconds_per_event(function, n):
    return 1e6 / n * min(timeit.repeat(
        lambda: function(params, path_times[:n], path_types[:n]), number=1, repeat=3
    ))


timings = pd.DataFrame(
    [(n, microseconds_per_event(total_intensities_a, n),
      microseconds_per_event(total_intensities_b, n))
     for n in (1_000, 2_000, 4_000, 8_000, 16_000)],
    columns=["events", "A (us per event)", "B (us per event)"],
).set_index("events")
timings["A / B"] = timings["A (us per event)"] / timings["B (us per event)"]
print(timings.round(2).to_string())

growth_a = timings["A (us per event)"].iloc[-1] / timings["A (us per event)"].iloc[0]
growth_b = timings["B (us per event)"].iloc[-1] / timings["B (us per event)"].iloc[0]
ratio = timings["A / B"]
print(f"time per event from n = 1000 to 16000: A x{growth_a:.1f}, B x{growth_b:.2f}; "
      f"A / B from {ratio.iloc[0]:.1f} to {ratio.iloc[-1]:.1f}")
verify("question.intensityByRecursionAgainstSum", "A", [
    growth_b < 1.5 and growth_a > 3 and bool((ratio > 1).all())
    and ratio.iloc[-1] > 5 and ratio.iloc[-1] > 3 * ratio.iloc[0],
    growth_a < 1.5 and growth_b < 1.5,
    bool((ratio < 1).all()),
    growth_a > 3 and growth_b > 3,
])

## `question.exactWaitOnGivenUniforms`: one step of the exact scheme, by hand

The step of the package is run on prescribed uniforms, which `Uniforms` hands to it in order.

In [ ]:
def run():
    params = config.asymmetric_pair_params()
    for u2 in (np.exp(-1.0), np.exp(-3.0)):
        simulator = ExponentialHawkes.from_state(params, [1.5, 1.0])
        simulator.rng = Uniforms([np.exp(-0.4), u2, 0.75])
        time, event_type = simulator.step()
        print(round(time, 3))


params = config.asymmetric_pair_params()
print("column sums of A:", params.excitation.sum(axis=0), " mu_bar:", params.baseline.sum())
print("log 2 / 4 =", np.log(2) / 4)
first, second = outcome(run).split("\n")
print(first, "|", second)
verify("question.exactWaitOnGivenUniforms", "D", [
    first == "0.5" and second == "0.5",
    first == "0.5" and second == "inf",
    first == "0.173" and second == "0.32",
    first == "0.173" and second == "0.5",
    first == "0.173" and second == "inf",
])

## `question.typeDrawnBeforeTheDecay`: the type drawn on the intensities of the last event

In [ ]:
class DrawFirst(ExponentialHawkes):
    def step(self):
        beta = self.params.decay
        excess = self.total_intensity - self._baseline_total
        u1, u2 = self.rng.random(2)
        wait_baseline = -np.log(u1) / self._baseline_total
        if excess <= 0.0:
            wait_excited = np.inf
        else:
            remaining = 1.0 + beta * np.log(u2) / excess
            wait_excited = (
                -np.log(remaining) / beta if remaining > 0.0 else np.inf
            )
        event_type = self._draw_type_and_jump()
        self._advance(min(wait_baseline, wait_excited))
        return self.time, event_type

In [ ]:
def run():
    params = config.asymmetric_pair_params()
    for simulator_class in (ExponentialHawkes, DrawFirst):
        simulator = simulator_class.from_state(params, [1.5, 1.0])
        simulator.rng = Uniforms([np.exp(-0.4), np.exp(-1.0), 0.75])
        time, event_type = simulator.step()
        print(event_type, simulator.decayed_counts.round(3).tolist())


params = config.asymmetric_pair_params()
at_event = params.baseline + params.excitation @ np.array([0.75, 0.5])
stale = params.baseline + params.excitation @ np.array([1.5, 1.0])
print("intensities at the event:", at_event, " cumulative", np.cumsum(at_event))
print("intensities of the last event:", stale, " cumulative", np.cumsum(stale))
first, second = outcome(run).split("\n")
print(first, "|", second)
verify("question.typeDrawnBeforeTheDecay", "E", [
    first == second == "1 [0.75, 1.5]",
    first == "1 [0.75, 1.5]" and second == "0 [1.75, 0.5]",
    first == "1 [0.75, 1.5]" and second == "0 [2.5, 1.0]",
    first == "0 [1.75, 0.5]" and second == "0 [1.25, 0.5]",
    first == "1 [0.75, 1.5]" and second == "0 [1.25, 0.5]",
])

## `question.clippedExcitationClockAddsEvents`: a defective clock clipped to a finite one

In [ ]:
class Clipped(ExponentialHawkes):
    def step(self):
        beta = self.params.decay
        excess = self.total_intensity - self._baseline_total
        u1, u2 = self.rng.random(2)
        wait_baseline = -np.log(u1) / self._baseline_total
        if excess <= 0.0:
            wait_excited = np.inf
        else:
            remaining = max(1.0 + beta * np.log(u2) / excess, 1e-12)
            wait_excited = -np.log(remaining) / beta
        self._advance(min(wait_baseline, wait_excited))
        return self.time, self._draw_type_and_jump()

In [ ]:
def run(simulator_class):
    params = HawkesParams(
        baseline=[0.5, 0.5],
        excitation=[[15.0, 10.0], [10.0, 15.0]],
        decay=50.0,
    )
    events = list(simulator_class(params, rng=4).events(2e4))
    print(round(len(events) / 2e4, 2))


params = HawkesParams(baseline=[0.5, 0.5], excitation=[[15.0, 10.0], [10.0, 15.0]], decay=50.0)
total_rate = params.stationary_intensity().sum()
error = standard_error_of_rate(params, 2e4)
print(f"rho {params.branching_ratio:.3f}, stationary rate {total_rate:.4f}, SE {error:.4f}")
print(f"-log(1e-12) / beta = {-np.log(1e-12) / params.decay:.4f} s")
package, clipped = float(outcome(run, ExponentialHawkes)), float(outcome(run, Clipped))
print(f"package: {package}, {(package - total_rate) / error:+.1f} SE;   "
      f"clipped: {clipped}, {(clipped - total_rate) / error:+.1f} SE")
close = lambda rate: abs(rate - total_rate) < 4 * error
below = lambda rate: rate < total_rate - 10 * error
above = lambda rate: rate > total_rate + 10 * error
verify("question.clippedExcitationClockAddsEvents", "B", [
    close(package) and close(clipped),
    close(package) and above(clipped),
    close(package) and below(clipped),
    below(package) and close(clipped),
])

## `question.selfAndCrossExcitingShareTheirTimes`: the wait sees only the column sums

In [ ]:
def run():
    self_exciting = config.self_exciting_pair_params()
    cross_exciting = config.cross_exciting_pair_params()
    paths = [
        np.array(list(ExponentialHawkes(params, rng=3).events(100.0)))
        for params in (self_exciting, cross_exciting)
    ]
    times = [path[:, 0] for path in paths]
    types = [path[:, 1] for path in paths]
    print(np.allclose(times[0], times[1]), np.array_equal(types[0], types[1]))


print("column sums of A:", config.self_exciting_pair_params().excitation.sum(axis=0),
      config.cross_exciting_pair_params().excitation.sum(axis=0))
printed = outcome(run)
print(printed)
verify("question.selfAndCrossExcitingShareTheirTimes", "C", [
    printed == "True True",
    printed == "False False",
    printed == "True False",
    printed == "ValueError",
])

## `question.eventsHoldTheCrossingEvent`: a run in two stretches against one

In [ ]:
class Dropping(ExponentialHawkes):
    def events(self, horizon):
        while True:
            time, event_type = self.step()
            if time > horizon:
                return
            yield time, event_type

In [ ]:
def run():
    params = config.one_way_pair_params()
    for simulator_class in (ExponentialHawkes, Dropping):
        simulator = simulator_class(params, rng=11)
        first = list(simulator.events(10.0))
        rest = list(simulator.events(40.0))
        whole = list(simulator_class(params, rng=11).events(40.0))
        print(first + rest == whole, len(whole) - len(first + rest))


first, second = outcome(run).split("\n")
print(first, "|", second)
negative = lambda line: line.split()[0] == "False" and int(line.split()[1]) < 0
verify("question.eventsHoldTheCrossingEvent", "A", [
    first == "True 0" and second == "False 1",
    first == second == "True 0",
    first == second == "False 1",
    negative(first) and negative(second),
])

## `question.seedsOfAnOrderFlowMonteCarlo`: an integer seed, a shared generator

In [ ]:
def run():
    params = config.weakly_exciting_pair_params()
    rng = np.random.default_rng(5)

    def path(seed_or_generator):
        simulator = ExponentialHawkes(params, rng=seed_or_generator)
        return list(simulator.events(10.0))

    print(path(5) == path(5), path(rng) == path(rng), path(5) == path(rng))


printed = outcome(run)
print(printed)
verify("question.seedsOfAnOrderFlowMonteCarlo", "B", [
    printed == "True False True",
    printed == "True False False",
    printed == "True True True",
    printed == "False False False",
])

## `question.transposedKernelChangesTheRate`: the records read row by row, and transposed

In [ ]:
def run():
    records = config.ASYMMETRIC_PAIR_PARAMS
    kernel = np.array([record["Kernel"] for record in records])
    for excitation in (kernel.reshape(2, 2), kernel.reshape(2, 2).T):
        params = HawkesParams(
            baseline=[0.2, 0.6], excitation=excitation, decay=4.0
        )
        events = list(ExponentialHawkes(params, rng=1).events(2e4))
        print(round(len(events) / 2e4, 2))


print(pd.DataFrame(config.ASYMMETRIC_PAIR_PARAMS).to_string(index=False))
asymmetric = config.asymmetric_pair_params()
transposed = HawkesParams(baseline=asymmetric.baseline, excitation=asymmetric.excitation.T,
                          decay=asymmetric.decay)
print("rho:", asymmetric.branching_ratio, transposed.branching_ratio)
print("lambda*:", asymmetric.stationary_intensity(), transposed.stationary_intensity())
errors = [standard_error_of_rate(p, 2e4) for p in (asymmetric, transposed)]
printed = outcome(run)
print(printed, "  standard errors", np.round(errors, 4))
rates = printed.split("\n")
near = lambda line, rate, error: line.replace(".", "", 1).isdigit() and abs(
    float(line) - rate) < 4 * error
verify("question.transposedKernelChangesTheRate", "B", [
    len(rates) == 2 and near(rates[0], 3, errors[0]) and near(rates[1], 3, errors[0]),
    len(rates) == 2 and near(rates[0], 3, errors[0]) and near(rates[1], 2, errors[1]),
    len(rates) == 2 and near(rates[0], 2, errors[1]) and near(rates[1], 3, errors[0]),
    "ValueError" in printed,
])